# 📦 Centro de Control de Inventario — Pronóstico de Demanda y Gestión de Componentes Críticos

**Planta de ensamblaje automotriz**

**Misión:** evitar paros de línea por desabasto de componentes críticos (asientos, tableros, faros)
mediante pronóstico de demanda y visualización proactiva del riesgo de inventario.

---

## 📐 Lógica de negocio: Punto de Reorden y Stock de Seguridad

- **`Lead_Time_Days`**: tiempo (en días) que tarda el proveedor en entregar un nuevo pedido desde
  que se coloca la orden.
- **`Safety_Stock_Units`**: colchón mínimo de inventario para absorber variabilidad de demanda o
  retrasos del proveedor sin detener la línea.
- **`Reorder_Point`**: nivel de inventario en el que se debe colocar una nueva orden de compra,
  calculado como:

  ```
  Reorder_Point = Lead_Time_Days × Demanda_Diaria_Promedio
  ```

  Es decir, el inventario mínimo necesario para **sobrevivir** el tiempo de entrega del proveedor
  sin tocar el stock de seguridad.

- **Cantidad sugerida a ordenar**: se usa una versión simplificada de **EOQ (Economic Order
  Quantity)**:

  ```
  EOQ = √( (2 × Demanda_Anual × Costo_de_Ordenar) / Costo_de_Mantener_por_Unidad )
  ```

  ajustada hacia arriba si el hueco proyectado entre el stock disponible y el stock de seguridad
  (al final del *lead time*) es mayor que el EOQ teórico — para garantizar que la orden sí evite
  el desabasto, no solo minimice costos.

## 🚦 Semáforo de riesgo

| Color | Condición |
|---|---|
| 🟢 Verde  | Stock actual ≥ Reorder_Point |
| 🟡 Ámbar  | Safety_Stock ≤ Stock actual < Reorder_Point |
| 🔴 Rojo   | Stock actual < Safety_Stock, **o** el pronóstico proyecta desabasto dentro del *lead time* |


In [1]:
# ==========================================================================
# LIBRERÍAS
# ==========================================================================
import numpy as np
import pandas as pd
import sqlite3
import json
from datetime import datetime, timedelta

import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)
pd.set_option('display.max_columns', None)

print("✅ Librerías cargadas correctamente")


✅ Librerías cargadas correctamente


## Simulación de Demanda con Estacionalidad

Se simula el consumo diario de **3 piezas críticas** durante **180 días**:

- **`Seat_A`** (asientos), **`Dash_B`** (tableros), **`Headlight_C`** (faros).

Patrones incluidos:

- **Pico de demanda a inicio de mes** (días 1–3 del mes): la planeación de producción libera
  órdenes grandes al arrancar el mes.
- **Caída los fines de semana**: la línea reduce/detiene turnos sábado y domingo.
- **Ruido aleatorio** para simular variabilidad real de planta.
- **Reposición cada 30 días**, pero con cantidad variable (a veces insuficiente) para generar
  riesgo real de desabasto — el proveedor no siempre repone al 100% de lo solicitado (recortes de
  producción, problemas logísticos, etc.). El proveedor de **`Dash_B`** tiene, además, un problema
  crónico de confiabilidad (entrega sistemáticamente por debajo de lo solicitado), lo cual genera
  intencionalmente un escenario de **riesgo real** para poner a prueba las alertas del dashboard.


In [2]:
# ==========================================================================
# CONFIGURACIÓN DE PIEZAS CRÍTICAS
# ==========================================================================
PARTS_CONFIG = {
    'Seat_A': {
        'nombre': 'Asiento Delantero (Set)', 'base_demand': 45, 'lead_time_days': 5,
        'safety_days': 3, 'initial_stock': 600, 'ordering_cost': 150, 'holding_cost_unit': 2.5,
    },
    'Dash_B': {
        'nombre': 'Tablero de Instrumentos', 'base_demand': 38, 'lead_time_days': 7,
        'safety_days': 4, 'initial_stock': 480, 'ordering_cost': 200, 'holding_cost_unit': 3.0,
    },
    'Headlight_C': {
        'nombre': 'Faro Delantero (Par)', 'base_demand': 50, 'lead_time_days': 4,
        'safety_days': 3, 'initial_stock': 520, 'ordering_cost': 120, 'holding_cost_unit': 2.0,
    },
}

N_DIAS_HISTORIA = 180
FECHA_FIN = datetime(2026, 8, 18)                       # "hoy" para efectos del dashboard
FECHA_INICIO = FECHA_FIN - timedelta(days=N_DIAS_HISTORIA - 1)
CICLO_REPOSICION_DIAS = 30
TARGET_REPOSICION_DIAS = 30      # la reposición intenta llevar el stock a ~30 días de demanda
DIAS_DESDE_ULTIMA_REPOSICION = 16  # "hoy" cae 16 días después de la última reposición programada

# Confiabilidad de entrega por proveedor: fracción de lo solicitado que realmente entrega
# (Dash_B tiene un proveedor con problemas crónicos de cumplimiento -> genera riesgo real)
RANGO_CONFIABILIDAD_ENTREGA = {
    'Seat_A': (0.75, 1.00),
    'Dash_B': (0.45, 0.75),
    'Headlight_C': (0.80, 1.05),
}


def simulate_demand_data(parts_config=PARTS_CONFIG, seed=RANDOM_SEED):
    """
    Genera el historial de consumo diario (180 días) y el nivel de inventario resultante
    para cada pieza crítica, incluyendo estacionalidad de demanda y reposiciones periódicas
    (a veces insuficientes, según la confiabilidad del proveedor) cada 30 días.
    """
    local_rng = np.random.default_rng(seed)
    fechas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq='D')
    n_dias = len(fechas)

    # Índices de reposición: se calculan hacia atrás desde "hoy" para que la última reposición
    # caiga siempre a DIAS_DESDE_ULTIMA_REPOSICION días del final (posición realista de ciclo).
    idx_ultima_reposicion = (n_dias - 1) - DIAS_DESDE_ULTIMA_REPOSICION
    indices_reposicion = set()
    k = idx_ultima_reposicion
    while k > 0:
        indices_reposicion.add(k)
        k -= CICLO_REPOSICION_DIAS

    registros = []

    for part_id, cfg in parts_config.items():
        stock = cfg['initial_stock']
        rango_confiabilidad = RANGO_CONFIABILIDAD_ENTREGA[part_id]

        for dia_idx, fecha in enumerate(fechas):
            # --- Estacionalidad: pico de inicio de mes ---
            factor_inicio_mes = 1.6 if fecha.day <= 3 else 1.0
            # --- Estacionalidad: caída de fin de semana ---
            factor_finde = 0.35 if fecha.weekday() >= 5 else 1.0
            # --- Ruido aleatorio (causas comunes de variabilidad) ---
            ruido = local_rng.normal(0, cfg['base_demand'] * 0.08)

            demanda = max(0.0, cfg['base_demand'] * factor_inicio_mes * factor_finde + ruido)
            demanda = round(demanda, 1)

            # --- Salida diaria de inventario (no puede bajar de 0 => backorder si falta) ---
            faltante = max(0.0, demanda - stock)
            stock = max(0.0, stock - demanda)

            # --- Reposición periódica del proveedor (cada 30 días), no siempre suficiente ---
            reposicion_qty = 0.0
            if dia_idx in indices_reposicion:
                objetivo_stock = TARGET_REPOSICION_DIAS * cfg['base_demand']
                cantidad_necesaria = max(0.0, objetivo_stock - stock)
                factor_entrega = local_rng.uniform(*rango_confiabilidad)
                reposicion_qty = round(cantidad_necesaria * factor_entrega, 0)
                stock += reposicion_qty

            registros.append({
                'Date': fecha,
                'Part_ID': part_id,
                'Demand': demanda,
                'Stockout_Units': round(faltante, 1),
                'Replenishment_Qty': reposicion_qty,
                'Stock_Level': round(stock, 1),
            })

    df = pd.DataFrame(registros).sort_values(['Part_ID', 'Date']).reset_index(drop=True)
    return df


df_demanda = simulate_demand_data()
print(f"✅ {len(df_demanda)} registros diarios simulados para {df_demanda['Part_ID'].nunique()} piezas "
      f"({N_DIAS_HISTORIA} días: {FECHA_INICIO.date()} → {FECHA_FIN.date()})")
df_demanda.tail(10)


✅ 540 registros diarios simulados para 3 piezas (180 días: 2026-02-20 → 2026-08-18)


,Date,Part_ID,Demand,Stockout_Units,Replenishment_Qty,Stock_Level
530,2026-08-09,Seat_A,20.6,0.0,0.0,974.1
531,2026-08-10,Seat_A,43.2,0.0,0.0,930.9
532,2026-08-11,Seat_A,46.1,0.0,0.0,884.8
533,2026-08-12,Seat_A,44.9,0.0,0.0,839.9
534,2026-08-13,Seat_A,43.4,0.0,0.0,796.5
535,2026-08-14,Seat_A,43.2,0.0,0.0,753.3
536,2026-08-15,Seat_A,18.0,0.0,0.0,735.3
537,2026-08-16,Seat_A,14.7,0.0,0.0,720.6
538,2026-08-17,Seat_A,44.5,0.0,0.0,676.1
539,2026-08-18,Seat_A,45.1,0.0,0.0,631.0


## Base de Datos SQLite de Maestros (`parts_master`)

Almacena, para cada `Part_ID`, los parámetros de planeación de inventario: `Lead_Time_Days`,
`Safety_Stock_Units` y `Reorder_Point` (calculado a partir de la demanda diaria promedio
observada en el histórico simulado).


In [3]:
# ==========================================================================
# BASE DE DATOS SQLITE — MAESTRO DE PIEZAS (parts_master)
# ==========================================================================
DB_PATH = 'inventory_control.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS parts_master')
cursor.execute('''
    CREATE TABLE parts_master (
        Part_ID             TEXT PRIMARY KEY,
        Part_Nombre         TEXT NOT NULL,
        Lead_Time_Days      INTEGER NOT NULL,
        Avg_Daily_Demand    REAL NOT NULL,
        Safety_Stock_Units  REAL NOT NULL,
        Reorder_Point       REAL NOT NULL,
        Ordering_Cost       REAL NOT NULL,
        Holding_Cost_Unit   REAL NOT NULL
    )
''')

registros_master = []
for part_id, cfg in PARTS_CONFIG.items():
    demanda_promedio = df_demanda.loc[df_demanda['Part_ID'] == part_id, 'Demand'].mean()
    safety_stock = round(cfg['safety_days'] * demanda_promedio, 1)
    reorder_point = round(cfg['lead_time_days'] * demanda_promedio, 1)   # Reorder_Point = Lead_Time * Avg_Daily_Demand

    registros_master.append((
        part_id, cfg['nombre'], cfg['lead_time_days'], round(demanda_promedio, 2),
        safety_stock, reorder_point, cfg['ordering_cost'], cfg['holding_cost_unit']
    ))

cursor.executemany('''
    INSERT INTO parts_master
    (Part_ID, Part_Nombre, Lead_Time_Days, Avg_Daily_Demand, Safety_Stock_Units, Reorder_Point, Ordering_Cost, Holding_Cost_Unit)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''', registros_master)

# --- Persistir histórico de demanda/inventario ---
df_demanda_sql = df_demanda.copy()
df_demanda_sql['Date'] = df_demanda_sql['Date'].astype(str)
df_demanda_sql.to_sql('demand_history', conn, if_exists='replace', index=False)

conn.commit()

df_parts_master = pd.read_sql('SELECT * FROM parts_master', conn)
print(f"✅ Base de datos '{DB_PATH}' creada — {len(df_parts_master)} piezas maestras, {len(df_demanda_sql)} registros de histórico")
df_parts_master


✅ Base de datos 'inventory_control.db' creada — 3 piezas maestras, 540 registros de histórico


,Part_ID,Part_Nombre,Lead_Time_Days,Avg_Daily_Demand,Safety_Stock_Units,Reorder_Point,Ordering_Cost,Holding_Cost_Unit
0,Seat_A,Asiento Delantero (Set),5,38.14,114.4,190.7,150.0,2.5
1,Dash_B,Tablero de Instrumentos,7,32.64,130.6,228.5,200.0,3.0
2,Headlight_C,Faro Delantero (Par),4,42.13,126.4,168.5,120.0,2.0


## Pronóstico de Demanda (Holt-Winters / Suavizado Exponencial)

Se utiliza **Holt-Winters (Suavizado Exponencial Triple)** con estacionalidad **aditiva semanal**
(`seasonal_periods=7`), ya que la demanda muestra un patrón claro entre semana vs. fin de semana.
Se entrena sobre los últimos 90 días de histórico (suficiente para capturar 12+ ciclos semanales)
y se proyecta un **horizonte extendido** (máx. entre 7 días y `Lead_Time_Days + 3`) para poder
evaluar el riesgo de desabasto dentro del tiempo de entrega del proveedor, aunque este sea mayor
a 7 días.


In [4]:
# ==========================================================================
# PRONÓSTICO DE DEMANDA — HOLT-WINTERS (SUAVIZADO EXPONENCIAL)
# ==========================================================================
DIAS_ENTRENAMIENTO = 90

def generar_pronostico(part_id, horizonte_dias):
    """
    Ajusta un modelo Holt-Winters (estacionalidad aditiva semanal) sobre los últimos
    `DIAS_ENTRENAMIENTO` días de demanda de la pieza y devuelve el pronóstico para
    los próximos `horizonte_dias`.
    """
    serie = (df_demanda[df_demanda['Part_ID'] == part_id]
             .sort_values('Date')
             .tail(DIAS_ENTRENAMIENTO)
             .set_index('Date')['Demand'])

    try:
        modelo = ExponentialSmoothing(
            serie, trend=None, seasonal='add', seasonal_periods=7,
            initialization_method='estimated'
        ).fit(optimized=True)
        pronostico = modelo.forecast(horizonte_dias)
    except Exception:
        # Respaldo robusto: suavizado exponencial simple si Holt-Winters no converge
        nivel = serie.ewm(alpha=0.3).mean().iloc[-1]
        pronostico = pd.Series([nivel] * horizonte_dias)

    pronostico = pronostico.clip(lower=0)
    fechas_pronostico = pd.date_range(serie.index.max() + timedelta(days=1), periods=horizonte_dias, freq='D')
    pronostico.index = fechas_pronostico
    return pronostico


# Prueba rápida para una pieza
ejemplo = generar_pronostico('Seat_A', 7)
print("Pronóstico de ejemplo — Seat_A (7 días):")
ejemplo.round(1)


Pronóstico de ejemplo — Seat_A (7 días):


2026-08-19    45.9
2026-08-20    45.0
2026-08-21    42.4
2026-08-22    14.5
2026-08-23    16.0
2026-08-24    46.0
2026-08-25    44.6
Freq: D, dtype: float64

## Lógica de Asesoramiento

1. Se proyecta el inventario día a día usando el pronóstico de demanda.
2. Si el inventario proyectado **al llegar el pedido** (es decir, dentro de `Lead_Time_Days`) cae
   por debajo del `Safety_Stock_Units` → se dispara una alerta crítica con los días exactos que
   restan hasta el desabasto y la cantidad sugerida a ordenar.
3. La cantidad sugerida usa el mayor valor entre el **EOQ simplificado** y el **hueco mínimo**
   necesario para que el inventario vuelva a superar el stock de seguridad al final del *lead
   time* (garantiza que la orden resuelva el riesgo, no solo optimice costo).


In [5]:
# ==========================================================================
# PROYECCIÓN DE INVENTARIO, SEMÁFORO DE RIESGO Y ALERTAS
# ==========================================================================

def proyectar_inventario(part_id, stock_actual, horizonte_dias):
    """Proyecta el nivel de inventario día a día restando el pronóstico de demanda."""
    pronostico = generar_pronostico(part_id, horizonte_dias)
    inventario_proyectado = stock_actual - pronostico.cumsum()
    return pronostico, inventario_proyectado


def calcular_eoq(demanda_diaria_promedio, ordering_cost, holding_cost_unit):
    """EOQ simplificado: EOQ = sqrt(2 * Demanda_Anual * Costo_Ordenar / Costo_Mantener_Unidad)."""
    demanda_anual = demanda_diaria_promedio * 365
    eoq = np.sqrt((2 * demanda_anual * ordering_cost) / holding_cost_unit)
    return eoq


def evaluar_riesgo_pieza(part_id):
    """
    Evalúa el nivel de riesgo de una pieza: color de semáforo, proyección de inventario,
    alerta crítica (si aplica) y cantidad sugerida a ordenar.
    """
    master = df_parts_master[df_parts_master['Part_ID'] == part_id].iloc[0]
    lead_time = int(master['Lead_Time_Days'])
    safety_stock = master['Safety_Stock_Units']
    reorder_point = master['Reorder_Point']
    demanda_promedio = master['Avg_Daily_Demand']

    stock_actual = df_demanda[df_demanda['Part_ID'] == part_id].sort_values('Date')['Stock_Level'].iloc[-1]

    horizonte = max(7, lead_time + 3)
    pronostico, inventario_proyectado = proyectar_inventario(part_id, stock_actual, horizonte)

    # Inventario proyectado exactamente al final del lead time
    stock_en_lead_time = float(inventario_proyectado.iloc[lead_time - 1]) if lead_time <= horizonte else float(inventario_proyectado.iloc[-1])

    # Días hasta el desabasto (primer día en que el inventario proyectado toca 0)
    dias_a_desabasto = None
    for i, valor in enumerate(inventario_proyectado, start=1):
        if valor <= 0:
            dias_a_desabasto = i
            break

    # --- Semáforo de riesgo ---
    if stock_actual < safety_stock or (dias_a_desabasto is not None and dias_a_desabasto <= lead_time):
        color_riesgo = 'Rojo'
    elif stock_actual < reorder_point:
        color_riesgo = 'Ámbar'
    else:
        color_riesgo = 'Verde'

    # --- Cantidad sugerida a ordenar (EOQ simplificado, ajustado a cubrir el hueco de riesgo) ---
    eoq = calcular_eoq(demanda_promedio, master['Ordering_Cost'], master['Holding_Cost_Unit'])
    hueco_necesario = max(0.0, safety_stock - stock_en_lead_time + reorder_point)
    cantidad_sugerida = int(np.ceil(max(eoq, hueco_necesario) / 10.0) * 10)  # redondeo a decenas

    # --- Alerta crítica ---
    alerta = None
    if stock_en_lead_time < safety_stock:
        x_dias = dias_a_desabasto if dias_a_desabasto is not None else lead_time
        alerta = (
            f"🚨 CRITICAL - Part {part_id} will run out of stock in {x_dias} days. "
            f"Lead time is {lead_time} days. Urgent order of {cantidad_sugerida} units required "
            f"to avoid production stoppage."
        )

    return {
        'Part_ID': part_id,
        'Stock_Actual': round(float(stock_actual), 1),
        'Reorder_Point': round(float(reorder_point), 1),
        'Safety_Stock': round(float(safety_stock), 1),
        'Lead_Time_Days': lead_time,
        'Stock_Proyectado_En_Lead_Time': round(stock_en_lead_time, 1),
        'Dias_A_Desabasto': dias_a_desabasto,
        'Color_Riesgo': color_riesgo,
        'EOQ_Simplificado': round(float(eoq), 1),
        'Cantidad_Sugerida': cantidad_sugerida,
        'Alerta': alerta,
        'Pronostico': pronostico,
        'Inventario_Proyectado': inventario_proyectado,
    }


print("✅ Funciones de proyección de inventario y asesoramiento definidas")


✅ Funciones de proyección de inventario y asesoramiento definidas


## Dashboard Interactivo de Planificación

El panel incluye:

- **Selector de Pieza** (`Dropdown`)
- **Histórico (60 días) + Pronóstico (barras + línea)**
- **Curva de Inventario Proyectado** con línea roja de `Safety_Stock`
- **Semáforo de Riesgo** (🟢/🟡/🔴)
- **Panel de alerta** con la cantidad exacta sugerida a ordenar


In [6]:
# ==========================================================================
# DASHBOARD INTERACTIVO — SELECTOR DE PIEZA + PRONÓSTICO + INVENTARIO + SEMÁFORO
# ==========================================================================

selector_pieza = widgets.Dropdown(
    options=[(f"{pid} — {cfg['nombre']}", pid) for pid, cfg in PARTS_CONFIG.items()],
    value='Seat_A',
    description='Pieza:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)

salida_dashboard = widgets.Output()

COLORES_SEMAFORO = {'Verde': '#16a34a', 'Ámbar': '#f59e0b', 'Rojo': '#dc2626'}


def construir_grafico_historico_pronostico(part_id, pronostico):
    """Barras con demanda real de los últimos 60 días + línea de pronóstico a 7 días."""
    df_hist = (df_demanda[df_demanda['Part_ID'] == part_id]
               .sort_values('Date').tail(60))
    pronostico_7d = pronostico.iloc[:7]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df_hist['Date'], y=df_hist['Demand'], name='Demanda real',
        marker_color='#3b82f6', opacity=0.85
    ))
    fig.add_trace(go.Scatter(
        x=pronostico_7d.index, y=pronostico_7d.values, mode='lines+markers',
        name='Pronóstico (7 días)', line=dict(color='#f97316', width=2.5)
    ))
    # Línea que conecta el último dato real con el primer punto del pronóstico
    fig.add_trace(go.Scatter(
        x=[df_hist['Date'].iloc[-1], pronostico_7d.index[0]],
        y=[df_hist['Demand'].iloc[-1], pronostico_7d.values[0]],
        mode='lines', line=dict(color='#f97316', width=2.5, dash='dot'), showlegend=False
    ))
    fig.update_layout(
        title=f'Demanda Histórica (60 días) + Pronóstico Holt-Winters (7 días) — {part_id}',
        xaxis_title='Fecha', yaxis_title='Unidades / día',
        height=380, margin=dict(t=50, b=40, l=50, r=20),
        template='plotly_white', barmode='overlay', legend=dict(orientation='h', y=1.12)
    )
    return fig


def construir_grafico_inventario(part_id, resultado):
    """Curva de inventario proyectado con línea roja de Safety_Stock y línea ámbar de Reorder_Point."""
    stock_actual = resultado['Stock_Actual']
    inventario_proyectado = resultado['Inventario_Proyectado']

    fechas = [df_demanda[df_demanda['Part_ID'] == part_id]['Date'].max()] + list(inventario_proyectado.index)
    valores = [stock_actual] + list(inventario_proyectado.values)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fechas, y=valores, mode='lines+markers', name='Inventario proyectado',
        line=dict(color='#0ea5e9', width=2.5), fill='tozeroy', fillcolor='rgba(14,165,233,0.10)'
    ))
    fig.add_hline(y=resultado['Safety_Stock'], line_color='#dc2626', line_dash='dash',
                   annotation_text=f"Safety Stock = {resultado['Safety_Stock']:.0f}", annotation_position='right')
    fig.add_hline(y=resultado['Reorder_Point'], line_color='#f59e0b', line_dash='dot',
                   annotation_text=f"Reorder Point = {resultado['Reorder_Point']:.0f}", annotation_position='right')
    fig.add_vline(x=fechas[1 + resultado['Lead_Time_Days'] - 1] if resultado['Lead_Time_Days'] <= len(fechas) - 1 else fechas[-1],
                   line_color='#6366f1', line_dash='dashdot',
                   annotation_text='Llegada de pedido (Lead Time)', annotation_position='top')

    fig.update_layout(
        title=f'Curva de Inventario Proyectado — {part_id}',
        xaxis_title='Fecha', yaxis_title='Unidades en stock',
        height=380, margin=dict(t=50, b=40, l=50, r=140),
        template='plotly_white', showlegend=False
    )
    return fig


def construir_semaforo_html(resultado):
    """Semáforo de riesgo como círculo de color con etiqueta (HTML/CSS)."""
    color = COLORES_SEMAFORO[resultado['Color_Riesgo']]
    return HTML(f"""
    <div style='display:flex;align-items:center;gap:14px;font-family:sans-serif;
                padding:14px 18px;background:#f8fafc;border-radius:10px;'>
      <div style='width:46px;height:46px;border-radius:50%;background:{color};
                  box-shadow:0 0 0 4px {color}22;flex-shrink:0;'></div>
      <div>
        <div style='font-size:1.05em;font-weight:700;color:{color}'>Riesgo: {resultado['Color_Riesgo'].upper()}</div>
        <div style='font-size:0.85em;color:#475569'>
          Stock actual: <b>{resultado['Stock_Actual']:.0f}</b> unidades &nbsp;·&nbsp;
          Reorder Point: <b>{resultado['Reorder_Point']:.0f}</b> &nbsp;·&nbsp;
          Safety Stock: <b>{resultado['Safety_Stock']:.0f}</b>
        </div>
      </div>
    </div>
    """)


def refrescar_dashboard(part_id):
    """Redibuja todos los componentes del dashboard para la pieza seleccionada."""
    with salida_dashboard:
        clear_output(wait=True)

        resultado = evaluar_riesgo_pieza(part_id)

        display(construir_semaforo_html(resultado))

        fig_hist = construir_grafico_historico_pronostico(part_id, resultado['Pronostico'])
        fig_inv = construir_grafico_inventario(part_id, resultado)
        display(widgets.VBox([go.FigureWidget(fig_hist), go.FigureWidget(fig_inv)]))

        if resultado['Alerta']:
            display(HTML(
                f"<div style='padding:12px 16px;background:#fef2f2;border-left:4px solid #dc2626;"
                f"border-radius:4px;margin-top:8px;color:#7f1d1d;font-family:sans-serif;font-weight:600'>"
                f"{resultado['Alerta']}</div>"
            ))
        else:
            display(HTML(
                "<div style='padding:12px 16px;background:#f0fdf4;border-left:4px solid #16a34a;"
                "border-radius:4px;margin-top:8px;color:#14532d;font-family:sans-serif'>"
                "✅ Sin riesgo de desabasto dentro del lead time. Inventario saludable.</div>"
            ))


def _on_cambio_pieza(change):
    if change['name'] == 'value':
        refrescar_dashboard(change['new'])


selector_pieza.observe(_on_cambio_pieza, names='value')

display(widgets.VBox([selector_pieza, salida_dashboard]))
refrescar_dashboard(selector_pieza.value)


## Reporte Ejecutivo del Centro de Control (JSON)

Se evalúan las 3 piezas críticas al cierre del análisis y se exporta un reporte con el nivel de
riesgo actual y la cantidad sugerida a ordenar por pieza, listo para ser consumido por un sistema
MRP/ERP o un dashboard corporativo.


In [7]:
# ==========================================================================
# REPORTE FINAL — RIESGO Y CANTIDAD SUGERIDA POR PIEZA (JSON)
# ==========================================================================

reporte_piezas = {}
for part_id in PARTS_CONFIG:
    r = evaluar_riesgo_pieza(part_id)
    reporte_piezas[part_id] = {
        'Part_ID': part_id,
        'Nombre': PARTS_CONFIG[part_id]['nombre'],
        'Stock_Actual': r['Stock_Actual'],
        'Reorder_Point': r['Reorder_Point'],
        'Safety_Stock': r['Safety_Stock'],
        'Lead_Time_Days': r['Lead_Time_Days'],
        'Stock_Proyectado_En_Lead_Time': r['Stock_Proyectado_En_Lead_Time'],
        'Dias_A_Desabasto': r['Dias_A_Desabasto'],
        'Nivel_Riesgo': r['Color_Riesgo'],
        'EOQ_Simplificado': r['EOQ_Simplificado'],
        'Cantidad_Sugerida_A_Ordenar': r['Cantidad_Sugerida'],
        'Alerta': r['Alerta'],
    }

piezas_en_riesgo_rojo = [p for p in reporte_piezas.values() if p['Nivel_Riesgo'] == 'Rojo']

reporte_final = {
    'Fecha_Analisis': FECHA_FIN.isoformat(),
    'Total_Piezas_Evaluadas': len(reporte_piezas),
    'Piezas_En_Riesgo_Critico': len(piezas_en_riesgo_rojo),
    'Detalle_Por_Pieza': reporte_piezas,
    'Generado_En': datetime.now().isoformat(),
}

with open('inventory_control_report.json', 'w', encoding='utf-8') as f:
    json.dump(reporte_final, f, indent=2, ensure_ascii=False)

print(f"📋 Piezas en riesgo crítico (🔴 Rojo): {len(piezas_en_riesgo_rojo)} de {len(reporte_piezas)}")
for p in reporte_piezas.values():
    emoji = {'Verde': '🟢', 'Ámbar': '🟡', 'Rojo': '🔴'}[p['Nivel_Riesgo']]
    print(f"  {emoji} {p['Part_ID']:<12} stock={p['Stock_Actual']:>6.0f}  "
          f"riesgo={p['Nivel_Riesgo']:<6} orden_sugerida={p['Cantidad_Sugerida_A_Ordenar']}")

print("✅ Reporte guardado en 'inventory_control_report.json'")
reporte_final


📋 Piezas en riesgo crítico (🔴 Rojo): 1 de 3
  🟢 Seat_A       stock=   631  riesgo=Verde  orden_sugerida=1300
  🔴 Dash_B       stock=    85  riesgo=Rojo   orden_sugerida=1270
  🟢 Headlight_C  stock=   617  riesgo=Verde  orden_sugerida=1360
✅ Reporte guardado en 'inventory_control_report.json'


{'Fecha_Analisis': '2026-08-18T00:00:00',
 'Total_Piezas_Evaluadas': 3,
 'Piezas_En_Riesgo_Critico': 1,
 'Detalle_Por_Pieza': {'Seat_A': {'Part_ID': 'Seat_A',
   'Nombre': 'Asiento Delantero (Set)',
   'Stock_Actual': 631.0,
   'Reorder_Point': 190.7,
   'Safety_Stock': 114.4,
   'Lead_Time_Days': 5,
   'Stock_Proyectado_En_Lead_Time': 467.3,
   'Dias_A_Desabasto': None,
   'Nivel_Riesgo': 'Verde',
   'EOQ_Simplificado': 1292.5,
   'Cantidad_Sugerida_A_Ordenar': 1300,
   'Alerta': None},
  'Dash_B': {'Part_ID': 'Dash_B',
   'Nombre': 'Tablero de Instrumentos',
   'Stock_Actual': 84.6,
   'Reorder_Point': 228.5,
   'Safety_Stock': 130.6,
   'Lead_Time_Days': 7,
   'Stock_Proyectado_En_Lead_Time': -97.7,
   'Dias_A_Desabasto': 3,
   'Nivel_Riesgo': 'Rojo',
   'EOQ_Simplificado': 1260.3,
   'Cantidad_Sugerida_A_Ordenar': 1270,
   'Alerta': '🚨 CRITICAL - Part Dash_B will run out of stock in 3 days. Lead time is 7 days. Urgent order of 1270 units required to avoid production stoppage.'},
  

---

## 📌 Conclusiones

Este notebook integra un flujo completo de **planeación de inventario impulsada por pronóstico**:
simulación de demanda con estacionalidad realista → maestro de piezas en SQLite → pronóstico
Holt-Winters → proyección de inventario contra `Safety_Stock` y `Reorder_Point` → semáforo de
riesgo → alerta accionable con cantidad exacta a ordenar (EOQ simplificado) → reporte ejecutivo en
JSON.

El objetivo final es que se pueda anticipar un paro de línea por falta de
componentes con suficiente antelación (dentro del *lead time* del proveedor) y con una acción
concreta y cuantificada a tomar.
